In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sqlite3

pd.options.display.max_columns = 999
pd.options.display.max_rows = 999

pd.set_option('max_colwidth', None)

In [2]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

In [3]:
#return first 5 and last 5 rows of dataframe
def p(df_):
    return pd.concat([df_.head(), df_.tail()])



In [4]:
def get_df_ts_check():
    #get injury report for 2010-11 thru 2017-18 sesasons (for player relinquished): df_inj1018r
    df_inj1018r = pd.read_csv(os.path.join('..', 'processed_data', 'A9injury_game_date_team_relinquished_player_2010_2018.csv'), parse_dates=['game_date'])

    
    #read in nba boxscore advanced data for seasons 2009-10 thru 2017-18: df_5bsa1018
    df_3gdgiti0618 = pd.read_csv(os.path.join('..', 'processed_data', '3game_id_game_date_team_id.csv'), parse_dates=['game_date'])
    df_3gdgi0618 = df_3gdgiti0618[['game_date', 'game_id']].drop_duplicates()

    df_5bsa1018 = pd.read_csv(os.path.join('..', 'processed_data', '5boxscoreadvancedv2_2009_2018.csv'))
    df_5bsa1018.PLAYER_NAME = df_5bsa1018.PLAYER_NAME.fillna('')
    df_5bsa1018 = pd.merge(df_3gdgi0618, df_5bsa1018.rename(columns={'GAME_ID':'game_id'}), on='game_id', how='inner')
    df_5bsa1018.MIN = df_5bsa1018.MIN.fillna('00:00')
    df_5bsa1018.MIN = pd.to_timedelta('00:' + df_5bsa1018.MIN).astype('timedelta64[s]').astype('float64') / 60
    df_5bsa1018.TEAM_ID = df_5bsa1018.TEAM_ID.fillna(0).astype('int64')
    
    df_gdinj1018 = pd.merge(df_inj1018r, df_5bsa1018.rename(columns={'PLAYER_NAME':'player'}), how='inner', on=['game_date', 'player'])
    
    df_gdinj1018[['E_OFF_RATING', 'OFF_RATING',
       'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT',
       'AST_TOV', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT',
       'EFG_PCT', 'TS_PCT', 'USG_PCT', 'E_USG_PCT', 'E_PACE', 'PACE',
       'PACE_PER40', 'POSS', 'PIE']] = df_gdinj1018[['E_OFF_RATING', 'OFF_RATING',
       'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT',
       'AST_TOV', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT',
       'EFG_PCT', 'TS_PCT', 'USG_PCT', 'E_USG_PCT', 'E_PACE', 'PACE',
       'PACE_PER40', 'POSS', 'PIE']].fillna(0)
    
    df_gdinj1018['ig_player_injury'] = 0
    df_gdinj1018.loc[df_gdinj1018.MIN != 0, 'ig_player_injury'] = 1
    
    
    df_3gdgiti0618 = pd.read_csv(os.path.join('..', 'processed_data', '3game_id_game_date_team_id.csv'), parse_dates=['game_date'])
    
    df_ts_check = pd.merge(df_3gdgiti0618[['game_id', 'season']].drop_duplicates().reset_index(drop=True), df_gdinj1018, on=['game_id', 'season'], how='inner')
    
    return df_ts_check

In [5]:
os.getcwd()

'/Users/pauljacob/Documents/GitHub/DSCapstone2/02_Data_Wrangling'

In [6]:
df_ts_check = get_df_ts_check()[['TEAM_ID', 'season', 'Team']].drop_duplicates().reset_index(drop=True)
print(df_ts_check.shape)
df_ts_check

(240, 3)


,TEAM_ID,season,Team
0,1610612748,2010-11,Heat
1,1610612749,2010-11,Bucks
2,1610612765,2010-11,Pistons
3,1610612753,2010-11,Magic
4,1610612737,2010-11,Hawks
5,1610612743,2010-11,Nuggets
6,1610612762,2010-11,Jazz
7,1610612744,2010-11,Warriors
8,1610612757,2010-11,Blazers
9,1610612764,2010-11,Wizards


In [7]:
seasons = set(df_ts_check['season'])
seasonsl = list(seasons)
seasonsl
seasonsl.sort(key = str)
df_seasons_adjacent = pd.DataFrame({'s': seasonsl[0:7], 'e':seasonsl[1:8]})
df_seasons_adjacent

,s,e
0,2010-11,2011-12
1,2011-12,2012-13
2,2012-13,2013-14
3,2013-14,2014-15
4,2014-15,2015-16
5,2015-16,2016-17
6,2016-17,2017-18


In [8]:
for row in range(7):
    start_ = df_seasons_adjacent.iloc[row,:]['s']
    end_ = df_seasons_adjacent.iloc[row,:]['e']
    print(start_ + ' and ' + end_ + ' ' +
          str(set(df_ts_check.Team[df_ts_check.season == start_]) - set(df_ts_check.Team[df_ts_check.season == end_])))




#Hornets not in 2013-14 season?
#Bobcats not in 2014-15 season?

2010-11 and 2011-12 set()
2011-12 and 2012-13 set()
2012-13 and 2013-14 {'Hornets'}
2013-14 and 2014-15 {'Bobcats'}
2014-15 and 2015-16 set()
2015-16 and 2016-17 set()
2016-17 and 2017-18 set()


In [9]:
for row in range(7):
    start_ = df_seasons_adjacent.iloc[row,:]['s']
    end_ = df_seasons_adjacent.iloc[row,:]['e']
    print(end_ + ' and ' + start_ + ' ' +
          str(set(df_ts_check.Team[df_ts_check.season == end_]) - set(df_ts_check.Team[df_ts_check.season == start_])))



#Pelicans appear in 2013-14 season
#Hornets arrive in 2014-15 season

2011-12 and 2010-11 set()
2012-13 and 2011-12 set()
2013-14 and 2012-13 {'Pelicans'}
2014-15 and 2013-14 {'Hornets'}
2015-16 and 2014-15 set()
2016-17 and 2015-16 set()
2017-18 and 2016-17 set()


In [10]:
datapath = os.path.join('..', 'processed_data')

if not os.path.exists(datapath):
    os.mkdir(datapath)

datapath_df = os.path.join(datapath, '10team_id_season_team_2010_2018.csv')
if not os.path.exists(datapath_df):
    df_ts_check.to_csv(datapath_df, index=False)

In [11]:
df = pd.read_csv(os.path.join('..', 'processed_data', '10team_id_season_team_2010_2018.csv'))
print(df.shape)
df

(240, 3)


,TEAM_ID,season,Team
0,1610612748,2010-11,Heat
1,1610612749,2010-11,Bucks
2,1610612765,2010-11,Pistons
3,1610612753,2010-11,Magic
4,1610612737,2010-11,Hawks
5,1610612743,2010-11,Nuggets
6,1610612762,2010-11,Jazz
7,1610612744,2010-11,Warriors
8,1610612757,2010-11,Blazers
9,1610612764,2010-11,Wizards
